# MIRT++ — KBS Expanded Experiment (Colab Runner)

**23 datasets × 11 resampling methods × 3 classifiers**

### Before you run:
1. Upload these two files to **Google Drive → `My Drive/MIRT_Experiment/`**:
   - `mirtplus.py`
   - `kbs_experiment/kbs_run_final.py`
2. Run all cells top-to-bottom.
3. Results are saved to **`My Drive/MIRT_Experiment/results/`** — so if the session drops, you can resume by re-running and the completed datasets are skipped automatically.

> **Estimated runtime:** ~3–5 hours on Colab CPU (free tier). Use *Runtime → Change runtime type → High-RAM* if available.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── CONFIGURE THIS ──────────────────────────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/MIRT_Experiment'   # folder you created on Drive
RESULTS_DIR  = f'{DRIVE_BASE}/results'
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted.')
print(f'Results will save to: {RESULTS_DIR}')

## Step 2 — Install dependencies

In [ ]:
# Colab already has sklearn, numpy, pandas, scipy.
# We only need to add these:
!pip install imbalanced-learn scikit-posthocs -q
print('Done.')

## Step 3 — Copy project files from Drive to Colab

In [ ]:
import shutil, os

# Copy mirtplus.py → /content/mirtplus.py  (so `import mirtplus` works)
shutil.copy(f'{DRIVE_BASE}/mirtplus.py', '/content/mirtplus.py')

# Copy experiment script → /content/kbs_run_final.py
shutil.copy(f'{DRIVE_BASE}/kbs_run_final.py', '/content/kbs_run_final.py')

# Verify
for f in ['/content/mirtplus.py', '/content/kbs_run_final.py']:
    size = os.path.getsize(f)
    print(f'  OK  {f}  ({size:,} bytes)')

## Step 4 — Run the experiment

This cell runs the full experiment. Results are saved to Drive after **every dataset** completes, so you can safely stop and resume anytime.

In [ ]:
import os, sys

# Tell the script where to write results (Google Drive)
os.environ['KBS_OUTPUT_DIR'] = RESULTS_DIR

# Make sure /content is on the path so `import mirtplus` resolves
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# Run the experiment script
exec(open('/content/kbs_run_final.py').read())

## Step 5 — View results

Run this at any time (even mid-experiment) to see what's completed so far.

In [ ]:
import pandas as pd

csv_path = f'{RESULTS_DIR}/kbs_results_raw.csv'
df = pd.read_csv(csv_path)

print(f'Total rows: {len(df):,}')
print(f'Datasets completed: {df["dataset"].nunique()}')
print(f'Datasets: {sorted(df["dataset"].unique().tolist())}')
print()

# Mean performance per resampler
summary = df.groupby('resampler')[['f1','gmean','auc']].mean().sort_values('gmean', ascending=False)
print('Mean performance (sorted by G-Mean):')
print(summary.round(4).to_string())

## Step 6 — Download results to your PC

Run this cell to download the raw results CSV directly.

In [ ]:
from google.colab import files
files.download(f'{RESULTS_DIR}/kbs_results_raw.csv')
files.download(f'{RESULTS_DIR}/kbs_timings.csv')

---
## Tips

| Situation | What to do |
|---|---|
| Session disconnects | Re-run all cells — completed datasets are skipped automatically |
| Want faster runtime | Runtime → Change runtime type → **High-RAM CPU** |
| Want to run overnight | Check *Stay connected* or use **Colab Pro** for background execution |
| Want to check progress mid-run | Open a second Colab tab, mount Drive, run Step 5 |
| `import mirtplus` error | Re-run Step 3 |
| Drive not mounted | Re-run Step 1 |